# Phase 5 screen B — the coronal plane, laterality, and the overfit lever

Screen A settled the schedule: **8 epochs, OneCycle, AMP**, worth +0.0592 stable-6 over run 1 on
fold 0. It also produced two things this screen follows up.

**The model overfits.** At 8 epochs train stable-6 is 0.979 against val 0.836; at 12 epochs train
0.9975 and val *falls* to 0.828. More epochs is spent. Augmentation is the lever, and the two labels
that lost ground under longer training — Fracture (prevalence 0.014) and Effusion — are the ones a
memorising model hurts first.

**Screen A's laterality test measured a no-op, and it was a design error rather than a null
result.** `_SERIES_PRIORITY[0]` is `("Sagittal", 1)` and `mirrors_in_plane` is true only for
Axial/Coronal, so at `max_series=1` the single series is sagittal and is never mirrored. Only the
257 studies (5.83%) with no sagittal fluid-sensitive series could be affected at all. **The
laterality fix is untested, not disproven** — and `max_series=2` is what tests it, because the
second entry in the priority list is coronal.

That makes b1 vs b2 the experiment that matters: same coronal series, mirrored wrong (as Phase 2
left them, 51.1% side coverage) versus mirrored right (98.5%). Run 1's gold transfer put **MCL at
0.5034 — random — and Lateral OA at 0.6132**, both coronal findings, against Effusion's 0.9391 on
sagittal. If coronal series help at all, those are the labels that should move.

Schedule is frozen at screen A's winner throughout, so every row here differs from `s3` in input
or augmentation only.

In [ ]:
import glob, hashlib, os, shutil, sys, time

GIT_SHA = 'phase5-screen-b'
FOLDS_PRIMARY_V2_SHA256 = 'cace8c45733ee7920a442fa4ae3f1db4a0d76401504e4729b49562d5eab52047'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]
PREPPED_DIRS = sorted(glob.glob('/kaggle/input/**/prepped', recursive=True))
assert len(PREPPED_DIRS) == 4, f'expected 4 prep-shard outputs, found {len(PREPPED_DIRS)}'

uid_to_npz = {}
for d in PREPPED_DIRS:
    for p in sorted(glob.glob(os.path.join(d, '*.npz'))):
        uid = os.path.splitext(os.path.basename(p))[0]
        assert uid not in uid_to_npz, f'duplicate artifact for {uid}'
        uid_to_npz[uid] = p
NPZ_ROOT = '/kaggle/working/prepped_all'
os.makedirs(NPZ_ROOT, exist_ok=True)
for uid, p in uid_to_npz.items():
    dst = os.path.join(NPZ_ROOT, f'{uid}.npz')
    if not os.path.exists(dst):
        os.symlink(p, dst)
print(f'{len(uid_to_npz)} prepped artifacts mounted')

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')
print('src/knee mounted from', SRC)

In [ ]:
import inspect
import torch

assert torch.cuda.is_available(), 'no GPU attached'
for i in range(torch.cuda.device_count()):
    major, minor = torch.cuda.get_device_capability(i)
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}, sm_{major}{minor}')
    assert (major, minor) >= (7, 0), f'GPU {i} is sm_{major}{minor} (P100?) -- need T4'
device = 'cuda'

# A stale src dataset killed screen A's first run 7.5 min in; assert the exact
# attributes this kernel depends on rather than trusting the mount.
from knee.train import train_one_epoch as _t
from knee.dataset import PreppedStudyDataset as _d, augment_volume as _a
assert {'scaler', 'scheduler'} <= set(inspect.signature(_t).parameters)
_p = inspect.signature(_d.__init__).parameters
assert 'sides' in _p and 'augment' in _p, 'src predates the sides/augment support'
print('src carries sides + augment + AMP')

In [ ]:
import numpy as np
import pandas as pd

from knee.infer import LABEL_COLUMNS
from knee.train import (Timer, evaluate, load_gold_holdout, log_experiment,
                        make_folds, train_one_epoch, train_val_split)
from knee.dataset import PreppedStudyDataset
from knee.metrics import macro_auc, paired_macro_auc_delta, per_label_auc
from knee.model import KneeModel

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
all_uids = sorted(train_df['StudyInstanceUID'].astype(str))
assert len(all_uids) == 4407

pseudo_path = glob.glob('/kaggle/input/**/pseudo_labels_qwen3_4b.csv', recursive=True)[0]
pseudo = pd.read_csv(pseudo_path)
labels_all = pseudo[['StudyInstanceUID'] + [f'score_{l}' for l in LABEL_COLUMNS]].copy()
labels_all.columns = ['StudyInstanceUID'] + LABEL_COLUMNS
_vals = labels_all[LABEL_COLUMNS].to_numpy(dtype=float)
assert np.isfinite(_vals).all() and (_vals >= 0).all() and (_vals <= 1).all()
labels_all['StudyInstanceUID'] = labels_all['StudyInstanceUID'].astype(str)
labels_eval = labels_all.copy()
labels_eval[LABEL_COLUMNS] = (_vals >= 0.5).astype(float)

folds = make_folds(all_uids, n_folds=5, seed=0)
_digest = hashlib.sha256(
    '\n'.join(f'{u},{folds[u]}' for u in sorted(folds)).encode()).hexdigest()
assert _digest == FOLDS_PRIMARY_V2_SHA256, 'fold assignment is not primary_v2'

gold_df = train_df[train_df['ACL'].notna()].reset_index(drop=True)
assert len(gold_df) == 58
gold_df[['StudyInstanceUID']].to_csv('gold_tmp.csv', index=False)
holdout = load_gold_holdout('gold_tmp.csv')

VAL_FOLD = 0
train_uids, val_uids = train_val_split(folds, val_fold=VAL_FOLD, exclude_uids=holdout)
assert not (set(train_uids) & set(val_uids))
assert not (set(train_uids) | set(val_uids)) & holdout

oof_dir = os.path.dirname(glob.glob('/kaggle/input/**/oof_pred.npy', recursive=True)[0])
base_pred = np.load(f'{oof_dir}/oof_pred.npy')
base_true = np.load(f'{oof_dir}/oof_true.npy')
base_uids = np.load(f'{oof_dir}/oof_uids.npy', allow_pickle=True).astype(str)
sel = np.array([folds[u] == VAL_FOLD for u in base_uids])
assert list(base_uids[sel]) == list(val_uids)
BASE_PRED, BASE_TRUE = base_pred[sel], base_true[sel]

STABLE = ["Medial Meniscus", "Lateral Meniscus", "Effusion", "Synovitis", "Baker's", "Contusion"]
SIX = [LABEL_COLUMNS.index(l) for l in STABLE]
# The laterality hypothesis is about MCL and the OA compartments, which are NOT in
# the stable six -- so this screen watches them explicitly rather than only the
# subset it makes ship/kill calls on.
CORONAL_WATCH = ["MCL", "Lateral OA", "Medial OA", "Medial Meniscus", "Lateral Meniscus"]

lat = pd.read_csv(glob.glob('/kaggle/input/**/laterality_geometry_check.csv', recursive=True)[0])
def _final(r):
    if r['tag_side'] in ('L', 'R'):
        return r['tag_side']
    return r['geom_side'] if r['geom_side'] in ('L', 'R') else None
lat['final'] = lat.apply(_final, axis=1)
SIDES = {str(u): s for u, s in zip(lat['StudyInstanceUID'], lat['final']) if s in ('L', 'R')}
assert len(SIDES) == 4341, len(SIDES)

rng = np.random.default_rng(0)
TRAIN_PROBE = sorted(rng.choice(train_uids, size=len(val_uids), replace=False).tolist())
print(f'fold {VAL_FOLD}: {len(train_uids)} train / {len(val_uids)} val; '
      f'{len(SIDES)} corrected sides; run 1 fold-0 stable-6 '
      f'{macro_auc(BASE_TRUE[:, SIX], BASE_PRED[:, SIX]):.4f}')

In [ ]:
N_SLICES, BATCH_SIZE, LR, EPOCHS = 16, 8, 1e-4, 8
EXPERIMENTS_CSV = '/kaggle/working/experiments.csv'
_HEADER = ('date,git_sha,config_hash,hypothesis,fold_set,seed,acl_auc,mcl_auc,'
           'medial_meniscus_auc,lateral_meniscus_auc,medial_oa_auc,lateral_oa_auc,'
           'pf_oa_auc,effusion_auc,synovitis_auc,bakers_auc,contusion_auc,fracture_auc,'
           'macro_auc,paired_delta,train_minutes,inference_seconds,promoted')
with open(EXPERIMENTS_CSV, 'w') as f:
    f.write(_HEADER + '\n')

# Cheapest first: a session that runs out of GPU still leaves ranked rows behind.
CONFIGS = [
    dict(name='b3_s1_aug',        max_series=1, lat=False, augment=True),
    dict(name='b1_s2_nolat',      max_series=2, lat=False, augment=False),
    dict(name='b2_s2_lat',        max_series=2, lat=True,  augment=False),
    dict(name='b4_s2_lat_aug',    max_series=2, lat=True,  augment=True),
]
print(f'{len(CONFIGS)} configs, all at {EPOCHS} epochs OneCycle+AMP (screen A winner)')

In [ ]:
rows = []

def run_config(cfg):
    """Train one config on fold 0 and append one experiments.csv row.

    Loaders are built per config because max_series changes the input volume;
    only the training loader ever augments -- val and probe must see identical
    pixels across configs or their scores are not comparable."""
    ms, use_lat, use_aug = cfg['max_series'], cfg['lat'], cfg['augment']
    sides = SIDES if use_lat else None

    def loader(uids, labels_df, shuffle=False, augment=False):
        ds = PreppedStudyDataset(uids, NPZ_ROOT, labels_df=labels_df, n_slices=N_SLICES,
                                 max_series=ms, sides=sides, augment=augment)
        return torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

    tl = loader(train_uids, labels_all, shuffle=True, augment=use_aug)
    vl = loader(val_uids, labels_eval)
    pl = loader(TRAIN_PROBE, labels_eval)

    torch.manual_seed(0)
    model = KneeModel(backbone_name='efficientnet_b0', num_labels=len(LABEL_COLUMNS),
                      pretrained=True).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    scaler = torch.amp.GradScaler('cuda', enabled=True)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=EPOCHS,
                                                steps_per_epoch=len(tl))

    timer = Timer()
    for ep in range(EPOCHS):
        with timer:
            loss = train_one_epoch(model, tl, opt, device=device, scaler=scaler, scheduler=sched)
        print(f"  {cfg['name']} epoch {ep+1}/{EPOCHS} loss {loss:.4f}")

    t0 = time.time()
    y_true, y_pred = evaluate(model, vl, device=device)
    infer_s = time.time() - t0
    probe_true, probe_pred = evaluate(model, pl, device=device)

    full = macro_auc(y_true, y_pred)
    six = macro_auc(y_true[:, SIX], y_pred[:, SIX])
    six_train = macro_auc(probe_true[:, SIX], probe_pred[:, SIX])
    delta, lo, hi = paired_macro_auc_delta(y_true[:, SIX], BASE_PRED[:, SIX], y_pred[:, SIX])
    aucs = per_label_auc(y_true, y_pred)
    by_label = {l: float(a) for l, a in zip(LABEL_COLUMNS, aucs)}

    print(f"{cfg['name']}: full {full:.4f} | stable-6 val {six:.4f} train {six_train:.4f} "
          f"(gap {six_train - six:+.4f}) | delta vs run1 {delta:+.4f} [{lo:+.4f}, {hi:+.4f}] "
          f"| {timer.minutes:.1f} min")
    print('   coronal watch: ' + '  '.join(f'{l} {by_label[l]:.4f}' for l in CORONAL_WATCH))

    log_experiment(EXPERIMENTS_CSV, git_sha=GIT_SHA, config_hash=f"screen_{cfg['name']}",
                   hypothesis=('Phase 5 screen B (fold 0, ranks candidates, promotes none): '
                               f"max_series={ms} laterality_v2={use_lat} augment={use_aug}, "
                               f'schedule fixed at screen A winner ({EPOCHS}ep OneCycle AMP)'),
                   fold_set=f'primary_v2_fold{VAL_FOLD}', seed=0, per_label_auc=by_label,
                   macro_auc=float(full), paired_delta=float(delta),
                   train_minutes=timer.minutes, inference_seconds=infer_s, promoted=False)

    np.save(f"/kaggle/working/pred_{cfg['name']}.npy", y_pred)
    rows.append(dict(name=cfg['name'], max_series=ms, lat=use_lat, aug=use_aug,
                     full=full, six=six, six_train=six_train, gap=six_train - six,
                     delta=delta, lo=lo, hi=hi, minutes=timer.minutes, infer_s=infer_s,
                     **{f'L_{l}': by_label[l] for l in CORONAL_WATCH}))
    pd.DataFrame(rows).to_csv('/kaggle/working/screen_b_summary.csv', index=False)
    del model, opt, scaler, sched
    torch.cuda.empty_cache()

for cfg in CONFIGS:
    run_config(cfg)

## Reading screen B

- **b1 vs b2** is the laterality question screen A failed to ask. Same coronal series either way;
  only the mirroring differs. Watch MCL and the OA compartments, not the stable-six macro — the
  hypothesis is about labels the stable six deliberately excludes.
- **b3 vs s3 (0.8362)** is augmentation alone. The number to watch is the train-val gap, not just
  val: if the gap narrows and val rises, the overfit diagnosis was right.
- **b4** is the shipping candidate if the parts compose.

In [ ]:
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print()
S3_SIX = 0.8362   # screen A winner, same fold, same baseline arm
for _, r in summary.iterrows():
    print(f"{r['name']:16s} stable-6 {r['six']:.4f}  vs s3 {r['six'] - S3_SIX:+.4f}  "
          f"gap {r['gap']:+.4f}")
lat_pair = summary.set_index('name')
if {'b1_s2_nolat', 'b2_s2_lat'} <= set(lat_pair.index):
    print('\nlaterality effect given coronal (b2 - b1):')
    for c in ['six', 'full'] + [c for c in summary.columns if c.startswith('L_')]:
        print(f"  {c:22s} {lat_pair.loc['b2_s2_lat', c] - lat_pair.loc['b1_s2_nolat', c]:+.4f}")